In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import numpy as np
import gzip
import upsetplot
from collections import defaultdict
import itertools

In [ ]:
poly_homopolymer_regions = [309,310,311,16179,16180,16181,16182,16183]
excluded_samples = ['ST002-1D_LUNG-pacbio-uwsc-group1']

In [ ]:
## read in mitoscope qc files
df_pb = pd.read_csv("../benchmark/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = df_pb['Sample'].str.split('-', expand=True)
df_pb['Age'] = np.where(df_pb['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df_ont = pd.read_csv("../benchmark/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)
df_ont['Age'] = np.where(df_ont['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df = pd.concat([df_pb, df_ont]).reset_index()
df['Donor_Tissue'] = df['Donor'] + "-" + df['Tissue']
df = df.sort_values(['Tissue', 'Donor', 'Center'])

df = df[~(df['Sample'].isin(excluded_samples))]
df['Tissue'] = df['Tissue'].str.split('_', expand=True)[1].str.capitalize()
df['Seq_Tech'] = np.where(df['Seq_Tech'] == 'pacbio', 'PacBio', 'ONT')

df

In [ ]:
## plot read count, coverage, length stats
sns.set_theme(style="ticks", context="talk", font_scale=0.7)

for category in ["Mito_Read_Count", "Mito_Coverage", "Nuclear_Coverage", "Mean_Read_Length"]:

    fig, ax = plt.subplots(layout='constrained', figsize=(7, 5))

    sns.boxplot(
        data=df,
        x="Donor_Tissue",
        y=category,
        hue="Seq_Tech",
        showfliers=True,
        palette="Set2",
        widths=0.3,
        legend=False,
        ax=ax
    )

    # Get original tick labels
    ticks = ax.get_xticklabels()
    labels = [t.get_text() for t in ticks]
    donors  = [l.split("-")[0] for l in labels]  
    tissues = ['\n\n' + l.split("-")[1] for l in labels]

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(donors, rotation=0)
    ax.set_ylabel(category)
    ax.set_xlabel("")

    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
    sec.tick_params('x', length=0)

    for tick in sec.get_xticklabels():
        tick.set_fontweight("bold")
        #tick.set_fontsize(14)

    midpoints = [0.5,2.5,3.5]
    for x in midpoints:
        ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

    #plt.yscale('log')
    print(tissues)
    #plt.savefig(f"plots/fig3-benchmark_tissue_QC_{category}.pdf", dpi=300)
    plt.show()

In [ ]:
## plot mtDNA CN
sns.set_theme(style="ticks", context="talk", font_scale=0.7)

for category in ["mtDNA_CN"]:

    fig, ax = plt.subplots(layout='constrained', figsize=(6, 5))

    sns.boxplot(
        data=df,
        x="Donor_Tissue",
        y=category,
        hue="Seq_Tech",
        showfliers=True,
        palette="Set2",
        widths=0.3,
        legend=True,
        ax=ax
    )

    # Get original tick labels
    ticks = ax.get_xticklabels()
    labels = [t.get_text() for t in ticks]
    donors  = [l.split("-")[0] for l in labels]  
    tissues = ['\n\n' + l.split("-")[1].split("_")[1] for l in labels]

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(donors, rotation=0)
    ax.set_ylabel("mtDNA Copy Number")
    ax.set_xlabel("")

    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
    sec.tick_params('x', length=0)

    for tick in sec.get_xticklabels():
        tick.set_fontweight("bold")
        #tick.set_fontsize(14)

    midpoints = [0.5,2.5,3.5]
    for x in midpoints:
        ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

    plt.yscale('log')
    print(tissues)
    print(donors)
    plt.savefig(f"plots/fig3-benchmark_tissue_QC_{category}.pdf", dpi=300)
    plt.show()

In [ ]:
## plots with individual GCC values shown
g = sns.catplot(
    data=df,
    x="Donor",
    y="Mito_Read_Count",
    col="Tissue",
    row="Seq_Tech",
    hue="Center",
    kind="bar",
    height=3,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharex=False,
    sharey=False
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5)) 
g.set_titles("{col_name} {row_name}")
g.set_axis_labels("", "Number of mtDNA reads")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=df,
    x="Donor",
    y="Mito_Coverage",
    col="Tissue",
    row="Seq_Tech",
    hue="Center",
    kind="bar",
    height=3,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharex=False,
    sharey=False
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5)) 
g.set_titles("{col_name} {row_name}")
g.set_axis_labels("", "Mean mtDNA Coverage")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=df,
    x="Donor",
    y="Mean_Read_Length",
    col="Tissue",
    row="Seq_Tech",
    hue="Center",
    kind="bar",
    height=3,
    aspect=0.8,
    palette="muted",
    legend_out=False,
    sharex=False,
    sharey=True
)
sns.move_legend(g, "center right", bbox_to_anchor=(1.1,0.5)) 
g.set_titles("{col_name} {row_name}")
g.set_axis_labels("", "Mean mtDNA Read Length")
plt.tight_layout()
plt.show()


In [ ]:
## plot mt coverage vs nuc coverage 
sns.set_theme(style="ticks", context="talk", font_scale=0.75)

plt.figure(figsize=(6,5))
sns.scatterplot(
    data=df,
    x="Nuclear_Coverage",
    y="Mito_Coverage",
    hue="Tissue",     
    palette="muted",
    s=70,
    style="Seq_Tech",
    alpha=1)
plt.xlabel("Nuclear Genome Coverage")
plt.ylabel("Mitochondria Genome Coverage")
plt.yscale("log")
#plt.xscale("log")
plt.title("")
plt.legend(title="", loc='upper right', fontsize=11)
plt.tight_layout()
plt.savefig("plots/suppl/benchmark_tissue_coverage.pdf", dpi=300)
plt.show()



In [ ]:
## generate combined df with variants called across all PB and ONT samples
def read_vcf(input_file):
    with gzip.open(input_file, 'rt') as fr:
        for line in fr:
            if line.startswith('#CHROM'):
                header = line.strip().lstrip('#').split('\t')
                break

    df = pd.read_csv(input_file, comment='#', sep='\t', compression='gzip', names=header)
    return df

snv_vcf_pb = read_vcf("../benchmark/pacbio/output/merged.mt.baldur.annotated.vcf.gz")
snv_vcf_ont = read_vcf("../benchmark/ont/output/merged.mt.baldur.annotated.vcf.gz")

snv_vcf_pb['ID'] = snv_vcf_pb[['CHROM', 'POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)
snv_vcf_ont['ID'] = snv_vcf_ont[['CHROM', 'POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)

filter_col = [col for col in snv_vcf_pb if col.startswith('ST00')]
snv_df_pb = pd.melt(
    snv_vcf_pb[["POS", "ID", "INFO"] + filter_col],
    id_vars=['POS', 'ID', "INFO"],
    var_name='Sample',
    value_name='Value'
)

filter_col = [col for col in snv_vcf_ont if col.startswith('ST00')]
snv_df_ont = pd.melt(
    snv_vcf_ont[["POS", "ID", "INFO"] + filter_col],
    id_vars=['POS', 'ID', "INFO"],
    var_name='Sample',
    value_name='Value'
)

snv_df = pd.concat([snv_df_pb, snv_df_ont])

snv_df[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = snv_df['Sample'].str.split('-', expand=True)
snv_df['Tissue'] = snv_df['Tissue'].str.split("_", expand=True)[1].str.capitalize()
snv_df['Donor_Tissue'] = snv_df['Donor'] + "_" + snv_df['Tissue']

snv_df['VEP'] = snv_df['INFO'].str.extract(r'CSQ=(.+);')
snv_df[['Consequence', 'Impact', 'Symbol', 'Biotype']] = snv_df['VEP'].str.split('|', expand=True)[[1,2,3,7]]

snv_df['MITOMAP'] = snv_df['INFO'].str.extract(r'MITOMAP=(.+);')
snv_df[['Gene.Name', 'Gene.Type', 'Amino.Acid.Change', 'GB.Freq.FL', 'GB.Freq.CR', 'GB.Seqs.FL', 'GB.Seqs.CR', 'Disease', 'Status', 'Additional.Annotations', 'MitoTIP']] = snv_df['MITOMAP'].str.split('|', expand=True)[[0,1,2,3,4,5,6,9,10,12,13]]

snv_df = snv_df[~(snv_df['Sample'].isin(excluded_samples))]
snv_df = snv_df[~(snv_df['POS'].isin(poly_homopolymer_regions))]

snv_df


In [ ]:
## more formatting + add sample specific and total replicate support
snv_df[['GT', 'ADF', 'ADR','HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS']] = snv_df['Value'].str.split(':', expand=True)
snv_df = snv_df.drop(columns=['Value', 'INFO'])
snv_df = snv_df[(snv_df['GT'] != '.') & (snv_df['GT'] != './.') & (snv_df['GT'] != '././.') & (snv_df['GT'] != './././.') & (snv_df['GT'] != '././././.')]
snv_df['AF'] = snv_df['HPL']

### note - this doesn't seem to be necessary for baldur, bcftools norm takes care as expected ###
def extract_element(row):
    if ',' in row['AF']:
        parts = row['AF'].split(',')
        if row['GT'] == '0/1':
            return parts[1]
        elif row['GT'] == '1/0':
            return parts[0]
    return row['AF']

snv_df['AF'] = snv_df.apply(extract_element, axis=1)
snv_df['AF'] = snv_df['AF'].astype(float)

snv_df[['MT', 'POS', 'REF', 'ALT']] = snv_df['ID'].str.split('-', expand=True)
snv_df['indel'] = snv_df.apply(lambda row: 'indel' if (len(row['REF']) > 1 or len(row['ALT']) > 1) else 'snv', axis=1)

snv_df['reps_sample_specific'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['ID']
    .transform('count')
)

snv_df['reps_total'] = (
    snv_df
    .groupby(['ID'])['ID']
    .transform('count')
)

snv_df['AF'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['AF']
    .transform('mean')
)

def custom_join(series):
    return ','.join(series.astype(str))

snv_df['reps_sample_specific_names'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['Sample']
    .transform(custom_join)
)

snv_df['reps_total_names'] = (
    snv_df
    .groupby(['ID'])['Sample']
    .transform(custom_join)
)

snv_df

In [ ]:
## retain variants that are present at least twice (across donors,tissues,centers,techs)
high_conf_snv_df = snv_df[snv_df['reps_total'] > 1]

low_conf_snv_df = snv_df[snv_df['reps_total'] == 1]
low_conf_snv_df

In [ ]:
## low conf indels
print(low_conf_snv_df[low_conf_snv_df['indel'] == 'indel'].sort_values('AF')['Center'].value_counts())
print(low_conf_snv_df[low_conf_snv_df['indel'] == 'snv']['Center'].value_counts())


In [ ]:
## collapse df with all variants (high+low) to lose GCC/tech reps
all_df = snv_df.drop_duplicates(subset=['ID', 'Donor', 'Tissue']).drop(columns=['Sample', 'Seq_Tech', 'Center', 'Group', 'VEP', 'MITOMAP', 'POS', 'GT', 'ADF', 'ADR', 'HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS', 'MT', 'REF', 'ALT'])
all_df['confidence'] = np.where(all_df['reps_total'] > 1, 'high', 'low')
all_df['heteroplasmy_category'] = np.where(all_df['AF'] > 0.95, 'homo', 'hetero')
all_df.to_csv('tables/benchmarking_variants_ALL.csv')
all_df

print(len(all_df[(all_df['heteroplasmy_category'] == 'hetero') & (all_df['confidence'] == 'high') & (all_df['indel'] == 'indel')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'hetero') & (all_df['confidence'] == 'low') & (all_df['indel'] == 'indel')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'homo') & (all_df['confidence'] == 'high') & (all_df['indel'] == 'indel')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'homo') & (all_df['confidence'] == 'low') & (all_df['indel'] == 'indel')]))

print(len(all_df[(all_df['heteroplasmy_category'] == 'hetero') & (all_df['confidence'] == 'high') & (all_df['indel'] == 'snv')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'hetero') & (all_df['confidence'] == 'low') & (all_df['indel'] == 'snv')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'homo') & (all_df['confidence'] == 'high') & (all_df['indel'] == 'snv')]))
print(len(all_df[(all_df['heteroplasmy_category'] == 'homo') & (all_df['confidence'] == 'low') & (all_df['indel'] == 'snv')]))

In [ ]:
## collapse hc variants to lose GCC/tech reps
collapsed = high_conf_snv_df.drop_duplicates(subset=['ID', 'Donor', 'Tissue']).drop(columns=['Sample', 'Seq_Tech', 'Center', 'Group', 'VEP', 'MITOMAP', 'POS', 'GT', 'ADF', 'ADR', 'HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS', 'AF', 'MT', 'REF', 'ALT'])
collapsed

In [ ]:
## format annotation columns
def custom_join(series):
    return ','.join(series.astype(str))

collapsed_snv_df = high_conf_snv_df.drop_duplicates(subset=['ID', 'Donor', 'Tissue']).drop(
    columns=['Sample', 'Seq_Tech', 'Center', 'Group', 'VEP', 'MITOMAP', 'POS', 
             'GT', 'ADF', 'ADR', 'HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS', 'MT', 'REF', 'ALT'])

collapsed_snv_df['heteroplasmy_category'] = np.where(collapsed_snv_df['AF'] > 0.95, 'homo', 'hetero')

collapsed_snv_df['Status'] = np.where(collapsed_snv_df['Status'] == 'Reported%3B lineage L & M marker, also hg IJK', 'Reported', collapsed_snv_df['Status'])
collapsed_snv_df['Status'] = np.where(collapsed_snv_df['Status'] == '', 'N/A', collapsed_snv_df['Status'])

collapsed_snv_df['Gene.Type'] = collapsed_snv_df['Gene.Type'].fillna('')
collapsed_snv_df['Gene.Name'] = collapsed_snv_df['Gene.Name'].fillna('')

collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == '', collapsed_snv_df['Biotype'], collapsed_snv_df['Gene.Type'])
collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == 'protein_coding', 'protein coding', collapsed_snv_df['Gene.Type'])

# for the one variant that is in a noncoding region 7515-7517 isn't being labeled by mitomap
collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == '', 'noncoding', collapsed_snv_df['Gene.Type'])

collapsed_snv_df['Consequence'] = (
    collapsed_snv_df['Consequence']
    .str.split('_')
    .apply(lambda x: '_'.join(x[:-1]))
)

indel_df = collapsed_snv_df[collapsed_snv_df['indel'] == 'indel']
collapsed_snv_df = collapsed_snv_df[collapsed_snv_df['indel'] == 'snv']

collapsed_snv_df


In [ ]:
indel_df

In [ ]:
## variants not in MITOMAP
collapsed_snv_df['in_MITOMAP'] = np.where(collapsed_snv_df['Gene.Name'] == '', 'No', 'Yes')
collapsed_snv_df[collapsed_snv_df['in_MITOMAP'] == 'No']['ID'].nunique()

In [ ]:
## write HC heteroplasmic SNVs
collapsed_snv_df[collapsed_snv_df['heteroplasmy_category'] == 'hetero'].to_csv('tables/benchmarking_snvs_hetero_only_HC.csv')

In [ ]:
collapsed_snv_df[collapsed_snv_df['Status'] == 'Cfrm [LP]'].sort_values(['AF'])

In [ ]:
## get df of region sizes
regions = pd.read_csv('jn_resources/GenomeLoci_MITOMAP_short.txt', sep='\t', names=['chrom', 'start', 'end', 'gene_name', 'gene_short_name', 'description'])
regions['size'] = regions['end'] - regions['start']

def classify(g):
    if g.startswith(("MT-ND", "MT-CO", "MT-ATP", "MT-CYB")):
        return "protein coding"
    elif g.startswith("MT-RNR"):
        return "rRNA"
    elif g.startswith("MT-T"):
        return "tRNA"
    elif g == "MT-CR":
        return "control region"
    else:
        return None

regions["biotype"] = regions["gene_name"].apply(classify)

region_sizes = regions.groupby('biotype')['size'].sum().reset_index()
region_sizes

In [ ]:
category_map = {
    'control region': 'intergenic',
    'noncoding': 'intergenic',
    'tRNA': 'non_coding_transcript_exon',
    'rRNA': 'non_coding_transcript_exon',
    'protein coding': 'protein coding'
}
region_sizes['for_conseq'] = region_sizes['biotype'].map(category_map)
regions_sizes_for_conseq = region_sizes.groupby('for_conseq')['size'].sum().reset_index()
regions_sizes_for_conseq

In [ ]:
## mitomap reported disease association counts
csq_counts_disease = collapsed_snv_df.groupby(['Donor', 'Tissue'])['Status'].value_counts().reset_index()
csq_counts_disease['Donor_Tissue'] = csq_counts_disease['Donor'] + "_" + csq_counts_disease['Tissue']

sns.set_theme(style="ticks", context="talk", font_scale=0.9)

g = sns.catplot(
    data=csq_counts_disease[csq_counts_disease['Status'] != 'N/A'].sort_values('Status', ascending=False),
    x='Status',
    hue='Donor_Tissue',
    y='count',
    kind='bar',
    height=5,
    aspect=1.75,
    sharex=False,
    legend=True
)

sns.move_legend(g, loc='lower left', bbox_to_anchor=(0.45,0.5), title="")
sns.despine(top=False, right=False, left=False, bottom=False)
g.set_axis_labels('', 'Variant Count')
g.set_titles('{col_name}') # Set titles for each facet

x_positions = [0, 1, 2, 3]
custom_labels = ['Reported', 'Conflicting\nReports', 'Cfrm [P]', 'Cfrm [LP]' ]
plt.xticks(x_positions, custom_labels)

plt.savefig(f"plots/fig4-variant_disease_associations.pdf", dpi=300)
plt.show()


In [ ]:
## gene type counts (normalized by region size)
csq_counts_type = collapsed_snv_df.groupby(['Donor', 'Tissue'])['Gene.Type'].value_counts().reset_index()
csq_counts_type['Donor_Tissue'] = csq_counts_type['Donor'] + "_" + csq_counts_type['Tissue']
csq_counts_type = csq_counts_type[csq_counts_type['Gene.Type'] != 'noncoding']
csq_counts_type = pd.merge(csq_counts_type, region_sizes, left_on="Gene.Type", right_on="biotype", how="left")
csq_counts_type['per_bp_in_region'] = csq_counts_type['count'] / csq_counts_type['size']

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=csq_counts_type[csq_counts_type['Gene.Type'] != 'noncoding'],
    x='Gene.Type',
    hue='Donor_Tissue',
    y='per_bp_in_region',
    kind='bar',
    height=5,
    aspect=2,
    sharex=True,
)

g.set_axis_labels("", r'Variant (bp$^{-1}$)')
#plt.xticks(rotation=90)
plt.show()

csq_counts_type.groupby(['Gene.Type']).agg(m=('per_bp_in_region', 'mean'))

In [ ]:
## add expected + obs/exp ratios
variant_count_totals = csq_counts_type.groupby(['Donor_Tissue']).agg(total_variants=('count', 'sum'))
csq_counts_type = pd.merge(csq_counts_type, variant_count_totals, on="Donor_Tissue", how="left")
csq_counts_type['expected_variant_freq'] = csq_counts_type['size'] / 16569
csq_counts_type['obs_variant_freq'] = csq_counts_type['count'] / csq_counts_type['total_variants']
csq_counts_type['obs_to_exp_ratio'] = csq_counts_type['obs_variant_freq'] / csq_counts_type['expected_variant_freq']

csq_counts_type

In [ ]:
## plot enriched/depleted gene type regions (CR, protein coding, tRNA, rRNA)
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

plt.figure(figsize=(6,5))

# Observed variants
sns.stripplot(
    data=csq_counts_type,
    x="Gene.Type",
    y="obs_variant_freq",
    order=['control region', 'protein coding', 'rRNA', 'tRNA'],
    hue="Tissue",
    s=8,
    legend=True,
    zorder=20  # on top
)

# Expected variants
sns.scatterplot(
    data=csq_counts_type,
    x="Gene.Type",
    y="expected_variant_freq",
    color='black',
    marker='X',
    s=80,
    label='Expected',
    zorder=30
)

sns.boxplot(
    x="Gene.Type",
    y="obs_variant_freq",
    order=['control region', 'protein coding', 'rRNA', 'tRNA'],
    data=csq_counts_type,
    color='white',
    linecolor='black',
    showfliers=False
)

# Custom x-axis labels
x_positions = [0, 1, 2, 3]
custom_labels = ['Control\nRegion', 'Protein\nCoding', 'rRNA', 'tRNA']
plt.xticks(x_positions, custom_labels)

plt.ylabel("Variant Frequency")
plt.xlabel("")

plt.legend()
plt.ylim(0,1)
plt.savefig(f"plots/fig4-variant_gene_types.pdf", dpi=300)
plt.show()


In [ ]:
csq_counts_type[['Gene.Type', 'expected_variant_freq']].drop_duplicates()

In [ ]:
## chi-square results for enrichemnt analysis
from scipy.stats import chisquare

obs = csq_counts_type.groupby("Gene.Type")["count"].sum()
lengths = region_sizes.set_index("biotype")["size"]
obs = obs.loc[lengths.index]
mt_length = lengths.sum()
expected = obs.sum() * (lengths / mt_length)

chi2, pval = chisquare(f_obs=obs, f_exp=expected)
print(chi2)
print(pval)


In [ ]:
## enrichment analysis results
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

total_variants = obs.sum()
results = []

for region in obs.index:
    k = obs[region]
    n = total_variants
    p = lengths[region] / mt_length
    
    # test enrichment/depletion
    p_enrich = binomtest(k, n, p, alternative='greater').pvalue
    p_deplete = binomtest(k, n, p, alternative='less').pvalue
    
    expected_k = n * p
    fold_enrich = k / expected_k
    fold_dep = expected_k / k

    
    results.append({
        "region": region,
        "observed": k,
        "expected": expected_k,
        "fold_enrichment": fold_enrich,
        "fold_depletion": fold_dep,
        "p_enrich": p_enrich,
        "p_deplete": p_deplete
    })

results_df = pd.DataFrame(results)

pvals_e = results_df["p_enrich"]  # or combine enrich/deplete carefully
pvals_d = results_df["p_deplete"]  # or combine enrich/deplete carefully

results_df["p_adj_e"] = multipletests(pvals_e, method='fdr_bh')[1]
results_df["p_adj_d"] = multipletests(pvals_d, method='fdr_bh')[1]

results_df

In [ ]:
## VEP consequence plot
csq_counts_conseq = collapsed_snv_df.groupby(['Donor', 'Tissue'])['Consequence'].value_counts().reset_index()
csq_counts_conseq['Donor_Tissue'] = csq_counts_conseq['Donor'] + "_" + csq_counts_conseq['Tissue']

category_map = {
    'intergenic': 'intergenic',
    'missense': 'protein coding',
    'synonymous': 'protein coding',
    'non_coding_transcript_exon': 'non_coding_transcript_exon',
}
csq_counts_conseq['mapped_category'] = csq_counts_conseq['Consequence'].map(category_map)
csq_counts_conseq = pd.merge(csq_counts_conseq, regions_sizes_for_conseq, left_on="mapped_category", right_on="for_conseq", how="left")
csq_counts_conseq['per_bp_in_region'] = csq_counts_conseq['count'] / csq_counts_conseq['size']
csq_counts_conseq


sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=csq_counts_conseq,
    x='Consequence',
    hue='Donor_Tissue',
    y='per_bp_in_region',
  #  col='',
    kind='bar',
    height=5,
    aspect=2,
    sharex=False,
)

g.set_axis_labels('', r'Variant (bp$^{-1}$)')
g.set_titles('{col_name}') # Set titles for each facet
#plt.xticks(rotation=90)
#plt.yticks([0,5,10,15,20])
plt.show()

csq_counts_conseq.groupby(['Consequence']).agg(m=('per_bp_in_region', 'mean'))

In [ ]:
## strand bias plot
collapsed_snv_df[['MT', 'POS', 'REF', 'ALT']] = collapsed_snv_df['ID'].str.split('-', expand=True)

collapsed_snv_df['nuc_change'] = collapsed_snv_df['REF'] + ">" + collapsed_snv_df['ALT']
collapsed_snv_df['rep_strand'] = np.where(collapsed_snv_df['REF'].isin(['C', 'T']), 'L-strand', 'H-strand')

conditions = [
    (collapsed_snv_df['nuc_change'] == 'G>A'),
    (collapsed_snv_df['nuc_change'] == 'G>T'),
    (collapsed_snv_df['nuc_change'] == 'G>C'),
    (collapsed_snv_df['nuc_change'] == 'A>T'),
    (collapsed_snv_df['nuc_change'] == 'A>C'),
    (collapsed_snv_df['nuc_change'] == 'A>G'),
]
values = ['C>T', 'C>A', 'C>G', 'T>A', 'T>G', 'T>C']

# Create a new column using np.select
collapsed_snv_df['nuc_change_revised'] = np.select(conditions, values, default='-')
collapsed_snv_df['nuc_change_revised'] = np.where(collapsed_snv_df['nuc_change_revised'] == '-', collapsed_snv_df['nuc_change'],collapsed_snv_df['nuc_change_revised'])


mut_sigs = collapsed_snv_df[collapsed_snv_df['Gene.Type'] != 'control region'].groupby(['Donor', 'Tissue', 'rep_strand'])['nuc_change_revised'].value_counts().reset_index()
mut_sigs['Donor_Tissue'] = mut_sigs['Donor'] + "_" + mut_sigs['Tissue']

mut_sigs["prop"] = (
    mut_sigs["count"]
    / mut_sigs.groupby("Donor_Tissue")["count"].transform("sum")
)

mut_sigs = mut_sigs.sort_values('nuc_change_revised')

mut_sigs

sns.set_theme(style="ticks", context="talk", font_scale=1)

g = sns.catplot(
    data=mut_sigs,
    x="nuc_change_revised",
    y="prop",
    kind="bar",
    hue="rep_strand",
    row="Tissue",       
    height=2,        
    aspect=3,
    width=0.5,
    sharey=True,      
    sharex=True,
    legend=True
)

# Format axes
g.set_axis_labels('Base Substitution', '')
g.set_titles("{row_name}")
#g.set(ylim=(0, 0.5))
sns.move_legend(g, "center right", bbox_to_anchor=(0.37,0.9), title="", fontsize=16) 
plt.tight_layout()
plt.savefig(f"plots/fig4-snv_base_subs.pdf", dpi=300)
plt.show()


In [ ]:
## HF histogram plot with break in middle

# 1. Generate sample data with two distinct ranges
data_low = collapsed_snv_df[collapsed_snv_df['heteroplasmy_category'] == 'hetero']["AF"]
data_high = collapsed_snv_df[collapsed_snv_df['heteroplasmy_category'] == 'homo']["AF"]
data = np.concatenate([data_low, data_high])

# Define the break points
xlim1_end = 0.1
xlim2_start = 0.9

# 2. Create subplots with shared y-axis
fig, (ax, ax2) = plt.subplots(1, 2, sharey=True, figsize=(5, 6), facecolor='w', gridspec_kw={'width_ratios': [1, 1]})
fig.subplots_adjust(wspace=0.5) # adjust space between axes

# 3. Plot the histogram on both axes with different x-limits
bins = 75 # Use a consistent number of bins across both plots
ax.hist(data, bins=bins, color='darkblue', edgecolor='black')
ax2.hist(data, bins=bins, color='skyblue', edgecolor='black')

# Zoom-in / limit the view to different portions of the data
ax.set_xlim(0, xlim1_end) # Main data range
ax2.set_xlim(xlim2_start, 1) # Outlier range

# 4. Hide the spines between ax and ax2 to create the "break" appearance
ax.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax.yaxis.tick_left()
ax2.yaxis.tick_right()
ax2.tick_params(labelleft=False, right=False) # remove the left tick labels on the right plot

# 5. Add the "cut-out" diagonal lines for visual indication of the break
d = .015 # how big to make the diagonal lines in axes coordinates
kwargs = dict(transform=ax.transAxes, color='k', clip_on=False)
ax.plot((1-d, 1+d), (-d, +d), **kwargs)
ax.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (1-d, 1+d), **kwargs)
ax2.plot((-d, +d), (-d, +d), **kwargs)

ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax2.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))

ax.set_ylabel("Variant Count", fontsize=14)
fig.supxlabel("    Heteroplasmy Level", y=0.001, fontsize=14)
fig.subplots_adjust(bottom=0.12, left=0.2)
plt.savefig(f"plots/fig4-heteroplasmy_levels_with_break.pdf", dpi=300)
plt.show()


In [ ]:

conditions = [
    (collapsed_snv_df['AF'] > 0.95),
    (collapsed_snv_df['AF'] > 0.05) & (collapsed_snv_df['AF'] <= 0.1),
    (collapsed_snv_df['AF'] > 0.01) & (collapsed_snv_df['AF'] <= 0.05),
    (collapsed_snv_df['AF'] <= 0.01)
]
values = ['homo', 'het_mid', 'het_low', 'het_ultra_low']

# Create a new column using np.select
collapsed_snv_df['het_group'] = np.select(conditions, values, default='Unknown')

collapsed_snv_df['het_group'].value_counts()


In [ ]:
# collapsed_snv_df.groupby(['Donor', 'Tissue', 'Donor_Tissue', 'heteroplasmy_category']).size().reset_index(name="count")
# sep_counts[sep_counts.heteroplasmy_category == 'hetero']

# snv_df['reps_sample_specific_names'] = (
#     snv_df
#     .groupby(['ID','Donor','Tissue'])['Sample']
#     .transform(custom_join)
# )

collapsed_snv_df[(collapsed_snv_df['Donor_Tissue'] == 'ST001_Lung') & (collapsed_snv_df['heteroplasmy_category'] == 'hetero')]['ID'].unique()


In [ ]:
## plot stacked bar plot with homo/hetero variant counts
all_counts = collapsed_snv_df.groupby(['Donor', 'Tissue', 'Donor_Tissue']).size().reset_index(name="count")
sep_counts = collapsed_snv_df.groupby(['Donor', 'Tissue', 'Donor_Tissue', 'heteroplasmy_category']).size().reset_index(name="count")
homo_counts = sep_counts[sep_counts.heteroplasmy_category == 'homo']

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

fig, ax = plt.subplots(layout='constrained', figsize=(5, 5))

sns.barplot(
    data=all_counts,
    x="Donor_Tissue",
    order=['ST001_Liver', 'ST001_Lung', 'ST002_Lung', 'ST002_Colon', 'ST003_Brain', 'ST004_Brain'],
    y="count",
    edgecolor='black',
    ax=ax,
    alpha=1,
    color="darkblue"
)
sns.barplot(
    data=homo_counts,
    x="Donor_Tissue",
    order=['ST001_Liver', 'ST001_Lung', 'ST002_Lung', 'ST002_Colon', 'ST003_Brain', 'ST004_Brain'],
    y="count",
    estimator=sum, 
    color="lightblue",
    alpha=1,
    edgecolor='black',
    ax=ax
)

# Get original tick labels
ticks = ax.get_xticklabels()
labels = [t.get_text() for t in ticks]
donors  = [l.split("_")[0] for l in labels]  
tissues = ['\n\n' + l.split("_")[1] for l in labels]

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(donors, rotation=0)
ax.set_ylabel("Variant Count")
ax.set_xlabel("")

sec = ax.secondary_xaxis(location=0)
sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
sec.tick_params('x', length=0)

for tick in sec.get_xticklabels():
    tick.set_fontweight("bold")
    #tick.set_fontsize(14)

midpoints = [0.5,2.5,3.5]
for x in midpoints:
    ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

# add legend
top_bar = mpatches.Patch(color='darkblue', label='Hetero')
bottom_bar = mpatches.Patch(color='skyblue', label='Homo')
plt.legend(handles=[top_bar, bottom_bar], fontsize=12)
plt.savefig(f"plots/fig4-het_vs_hom_variant_counts.pdf", dpi=300)
print(tissues)
print(donors)

In [ ]:
collapsed_snv_df.to_csv(f"tables/high_conf_snvs.csv", index=False)

In [ ]:
collapsed_snv_df[collapsed_snv_df['heteroplasmy_category'] == 'hetero']['Donor_Tissue'].value_counts()

In [ ]:
## distribution of HF values across samples
hetero_snvs = collapsed_snv_df[collapsed_snv_df['heteroplasmy_category'] == 'hetero'].sort_values(['Donor', 'Tissue'])

sns.set_theme(style="ticks", context="talk", font_scale=0.9)
plt.figure(figsize=(7,5))

sns.boxplot(hetero_snvs, 
            x='Donor_Tissue', 
            y='AF', 
            hue='Donor_Tissue', 
            showmeans=True, 
            meanprops={
                "marker": "x",
                "markerfacecolor": "white", 
                "markeredgecolor": "black", 
                "markersize": "6"}, 
            showfliers=False)

plt.xticks(rotation=90)
#plt.ylim(0, 0.04)
plt.xlabel('')
plt.ylabel('Heteroplasmic Frequency')
plt.savefig(f"plots/fig4-HF_by_tissue.pdf", dpi=300,  bbox_inches='tight')
plt.show()


In [ ]:
## heatmaps per donor with reps collapsed
sns.set_theme(style="ticks", context="talk", font_scale=0.5)

donors = ['ST001', 'ST002', 'ST003', 'ST004']
tissues = snv_df['Tissue'].unique()

for d in donors:

    heatmap_data = collapsed_snv_df[(collapsed_snv_df['Donor'] == d)].pivot(index=['Donor','Tissue'], columns=['ID','POS'], values='AF')

    ts = collapsed_snv_df[collapsed_snv_df['Donor'] == d]['Tissue'].unique()

    # Sort columns by highest average AF
    #heatmap_data = heatmap_data.sort_index(axis=1, level=[1,0], ascending=True)
    #column_order = heatmap_data.mean(axis=0).sort_values(ascending=False).index
    column_order = heatmap_data.sort_values(by=(d, ts[0]),axis=1, ascending=False).columns
    heatmap_data = heatmap_data[column_order]

    # Plot heatmap
    plt.figure(figsize=(6, 16))
    sns.heatmap(heatmap_data.T, cmap="viridis", linewidths=1,annot=False, cbar_kws={'label': 'Heteroplasmy Level'})
    plt.title('')
    plt.xlabel('')
    plt.ylabel('')
    plt.tight_layout()

    #plt.savefig(f'plots/benchmark_tissues.snv_heatmap.{d}.png', dpi=300)
    plt.show()


In [ ]:
## individual heatmaps per sample with GCC/tech replicates
sns.set_theme(style="ticks", context="talk", font_scale=0.5)

donors = ['ST001', 'ST002', 'ST003', 'ST004']
donor_tissues = snv_df['Donor_Tissue'].unique()

for d in donor_tissues:

    heatmap_data = high_conf_snv_df[high_conf_snv_df['Donor_Tissue'] == d].pivot(index=['Sample','Donor_Tissue'], columns='ID', values='AF')

    # Sort columns by highest average AF
    column_order = heatmap_data.mean(axis=0).sort_values(ascending=False).index
    heatmap_data = heatmap_data[column_order]

    # Plot heatmap
    plt.figure(figsize=(16, 6))
    sns.heatmap(heatmap_data, cmap="viridis", linewidths=1,annot=False, cbar_kws={'label': 'Heteroplasmy Level'})
    plt.title('Heteroplasmy Heatmap')
    plt.xlabel('Variant')
    plt.ylabel('Sample')
    plt.tight_layout()
    plt.show()



In [ ]:
## combined heatmap of SNVs with collapsed reps
sns.set_theme(style="ticks", context="talk", font_scale=0.5)

donors = ['ST001', 'ST002', 'ST003', 'ST004']
tissues = snv_df['Tissue'].unique()

heatmap_data = collapsed_snv_df.pivot(index=['Donor','Tissue'], columns=['ID','POS'], values='AF')
ts = collapsed_snv_df[collapsed_snv_df['Donor'] == d]['Tissue'].unique()

# Sort columns by highest average AF
#heatmap_data = heatmap_data.sort_index(axis=1, level=[1,0], ascending=True)
column_order = heatmap_data.mean(axis=0).sort_values(ascending=False).index
#column_order = heatmap_data.sort_values(by=(d, ts[0]),axis=1, ascending=False).columns
heatmap_data = heatmap_data[column_order]

# Plot heatmap
plt.figure(figsize=(6, 16))
sns.heatmap(heatmap_data.T, cmap="viridis", linewidths=1,annot=False, cbar_kws={'label': 'Heteroplasmy Level'})
plt.title('')
plt.xlabel('')
plt.ylabel('')
plt.tight_layout()

#plt.savefig(f'plots/benchmark_tissues.snv_heatmap.pdf', dpi=300)
plt.show()



In [ ]:
## homo/hetero upset plots
samples = collapsed_snv_df['Donor_Tissue'].unique()

new_df = collapsed_snv_df.groupby(['ID', 'heteroplasmy_category'])['Donor_Tissue'].agg(','.join).reset_index()

upset_df_homo = upsetplot.from_memberships(new_df[new_df['heteroplasmy_category'] == 'homo'].Donor_Tissue.str.split(","), data=new_df[new_df['heteroplasmy_category'] == 'homo'])
upset_df_hetero = upsetplot.from_memberships(new_df[new_df['heteroplasmy_category'] == 'hetero'].Donor_Tissue.str.split(","), data=new_df[new_df['heteroplasmy_category'] == 'hetero'])

upset_df_homo['n_donors'] = (
    upset_df_homo['Donor_Tissue']
    .str.findall(r'ST\d+')
    .apply(lambda x: len(set(x)))
)
upset_df_homo = upset_df_homo.sort_values(['n_donors', 'Donor_Tissue'], ascending=[True,False])

upset_df_hetero['n_donors'] = (
    upset_df_hetero['Donor_Tissue']
    .str.findall(r'ST\d+')
    .apply(lambda x: len(set(x)))
)
upset_df_hetero['intersection_size'] = upset_df_hetero['Donor_Tissue'].str.split(',').apply(len)
upset_df_hetero = upset_df_hetero.sort_values(['intersection_size', 'n_donors', 'Donor_Tissue'], ascending=[True, True,False])

color_map = {
    1: "blue",
    2: "green",
    3: "purple",
    4: "red"
}

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig = plt.figure(figsize=(10,6))
t = upsetplot.UpSet(upset_df_homo, element_size=None, sort_categories_by="-input", sort_by="input", show_percentages=True, totals_plot_elements=0)

for r in range(1, len(samples) + 1):
    for present_tuple in itertools.combinations(samples, r):

        present = list(present_tuple)
        absent = [s for s in samples if s not in present]

        donors = {s.split('_')[0] for s in present}
        n_donors = len(donors)

        t.style_subsets(facecolor=color_map[n_donors], present=present, absent=absent)

t.plot(fig)
plt.suptitle("Homoplasmic", size=16)
fig.savefig("plots/fig4-upset_plot_homo.pdf", dpi=300)
plt.show()


sns.set_theme(style="ticks", context="talk", font_scale=0.65)
fig = plt.figure(figsize=(14,6))
t = upsetplot.UpSet(upset_df_hetero, element_size=None, sort_categories_by="-input", sort_by="input", show_percentages=True, totals_plot_elements=0)

t.style_subsets(facecolor='blue', min_degree=1, max_degree=1)
t.style_subsets(facecolor='green', min_degree=2, max_degree=2)
t.style_subsets(facecolor='orange', min_degree=3, max_degree=3)
t.style_subsets(facecolor='purple', min_degree=4, max_degree=4)
t.style_subsets(facecolor='red', min_degree=5, max_degree=5)

t.plot(fig)
plt.suptitle("Heteroplasmic", size=16)
fig.savefig("plots/fig4-upset_plot_hetero.pdf", dpi=300)
plt.show()


In [ ]:
## pileup verification for discordant positions (8860,4769)
positions = [66, 3243, 7515, 13042]
indir="pileup_bases/benchmark"

comb_x = pd.DataFrame()
for s in df['Sample'].unique():
    for p in positions:
        x = pd.read_csv(f'{indir}/{s}.{p}.base_comp.txt', sep=" ", names=['freq', 'base'])
        x['sample'] = s
        x['pos'] = p
        comb_x = pd.concat([comb_x, x])

sr_data = pd.read_csv('../../../mutect2/smaht/illumina/benchmark/samples_full.csv', names=['sample_id', 'file'])
for s in sr_data['sample_id'].unique():
    for p in positions:
        x = pd.read_csv(f'{indir}/{s}.{p}.base_comp.txt', sep=" ", names=['freq', 'base'])
        x['sample'] = s
        x['pos'] = p
        comb_x = pd.concat([comb_x, x])
    
comb_x['base'] = comb_x['base'].str.upper()
#comb_x = comb_x[~comb_x['base'].isin(['*'])]

comb_x_collapsed = comb_x.groupby(['sample', 'pos', 'base'])['freq'].sum().reset_index()
comb_x_collapsed['prop'] = comb_x_collapsed['freq'] / comb_x_collapsed.groupby(['sample', 'pos'])['freq'].transform('sum')
comb_x_collapsed[['donor', 'tissue', 'tech', 'center']] = comb_x_collapsed['sample'].str.split('-', expand=True)
comb_x_collapsed['Donor_Tissue'] = comb_x_collapsed['donor'] + "-" + comb_x_collapsed['tissue']
comb_x_collapsed

idx_levels = {col: comb_x_collapsed[col].unique() for col in ['sample', 'pos', 'base']}
new_idx = pd.MultiIndex.from_product(idx_levels.values(), names=idx_levels.keys())
comb_x_collapsed = comb_x_collapsed.set_index(list(idx_levels)).reindex(new_idx, fill_value=0).reset_index()
comb_x_collapsed[['donor', 'tissue', 'tech', 'center']] = comb_x_collapsed['sample'].str.split('-', expand=True)
comb_x_collapsed['Donor_Tissue'] = comb_x_collapsed['donor'] + "-" + comb_x_collapsed['tissue']
comb_x_collapsed


In [ ]:
## plot pileup results 
sns.set_theme(style="ticks", context="talk", font_scale=0.7)

# Use sns.catplot() instead of sns.barplot()
g = sns.catplot(
    data=comb_x_collapsed[(comb_x_collapsed['pos'] == 3243 ) & (comb_x_collapsed['prop'] < 0.95) & (~(comb_x_collapsed['base'].isin([',', '.', 'A'])))],
    x='base',
    order=['A', 'G', 'T', 'C'],
    y='prop',
    hue='tech',
    col='Donor_Tissue',
    col_wrap=6,
    kind='strip',
    height=2.5,
    aspect=1,
    dodge=True,
    s=50
)

g.set_axis_labels("Base", "Proportion")
#for ax in g.axes.flat:
   # ax.set_yscale('log')
   # ax.set_ylim(0,0.01)
g.set_titles("{col_name}")
plt.savefig(f"plots/suppl/pileup_freqs_pathogenic_variants.3243.pdf", dpi=300)
plt.show()


# Use sns.catplot() instead of sns.barplot()
g = sns.catplot(
    data=comb_x_collapsed[(comb_x_collapsed['pos'] == 13042 ) & (comb_x_collapsed['prop'] < 0.95) & (~(comb_x_collapsed['base'].isin([',', '.', 'G'])))],
    x='base',
    order=['A', 'G', 'T', 'C'],
    y='prop',
    hue='tech',
    col='Donor_Tissue',
    col_wrap=6,
    kind='strip',
    height=2.5,
    aspect=1,
    dodge=True,
    s=50
)

g.set_axis_labels("Base", "Proportion")
#for ax in g.axes.flat:
   # ax.set_yscale('log')
   # ax.set_ylim(0,0.01)
g.set_titles("{col_name}")
plt.savefig(f"plots/suppl/pileup_freqs_pathogenic_variants.13042.pdf", dpi=300)
plt.show()

# Use sns.catplot() instead of sns.barplot()
g = sns.catplot(
    data=comb_x_collapsed[(comb_x_collapsed['pos'] == 7515 ) & (comb_x_collapsed['prop'] <= 1) & (~comb_x_collapsed['base'].isin(['*', ',', '.', 'A']))],
    x='base',
  #  order=['A', 'G', 'T', 'C'],
    y='prop',
    hue='tech',
    col='Donor_Tissue',
    col_wrap=3,
    kind='strip',
    height=3.5,
    aspect=1,
    dodge=True,
    s=50
)

g.set_axis_labels("Base", "Proportion")
#for ax in g.axes.flat:
   # ax.set_yscale('log')
   # ax.set_ylim(0,0.01)
g.set_titles("{col_name}")
#plt.savefig(f"plots/supplementary/pileup_freqs_pathogenic_variants.13042.pdf", dpi=300)
plt.show()




In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.7)

# Use sns.catplot() instead of sns.barplot()
g = sns.catplot(
    data=comb_x_collapsed[(comb_x_collapsed['pos'] == 66) &
                          (comb_x_collapsed['tech'] != 'illumina') &
                          (comb_x_collapsed['prop'] <= 1) &
                          (comb_x_collapsed['Donor_Tissue'] == 'ST001-1A_LIVER') &
                          (~comb_x_collapsed['base'].isin(['G', '.', ',', '*']))],
    x='base',
  #  order=['A', 'G', 'T', 'C'],
    y='prop',
    hue='center',
    hue_order=['bcm', 'broad', 'nygc', 'uwsc', 'washu'],
    col='Donor_Tissue',
   # col_wrap=3,
    row='tech',
    kind='strip',
    height=3.5,
    aspect=1.2,
    dodge=True,
    s=50
)

g.set_axis_labels("Base", "Proportion")
#for ax in g.axes.flat:
   # ax.set_yscale('log')
   # ax.set_ylim(0,0.01)
g.set_titles("{row_name}")
plt.savefig(f"plots/supplementary/lowfreqhetero.ST001-Liver.66G>A.pdf", dpi=300)
plt.show()

In [ ]:
comb_x_collapsed[(comb_x_collapsed['pos'] == 66) &
                          (comb_x_collapsed['tech'] != 'illumina') &
                          (comb_x_collapsed['prop'] <= 1) &
                          (comb_x_collapsed['Donor_Tissue'] == 'ST001-1A_LIVER')]